$$
D_{\mathrm{KL}}(p \,||\, q) = \int_{\mathcal{X}} p(x) \log \frac{p(x)}{q(x)} \, dx
$$

Not symmetric: $ D_{\mathrm{KL}}(p||q) \neq D_{\mathrm{KL}}(q||p) $  

- $ D_{\mathrm{KL}}(p||q) $ penalizes missing support $\to$ **mode covering**.

- $ D_{\mathrm{KL}}(q||p) $ penalizes assigning mass to unsupported regions $\to$ **mode seeking**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import Ellipse


In [ ]:
mu_p = np.array([2.0, -1.0])
Sigma_p = np.array([[5.4, 1.1],
                    [1.1, 1.5]])

Sigma_p_inv = np.linalg.inv(Sigma_p)
logdet_Sigma = np.log(np.linalg.det(Sigma_p))
tr_Sigma = np.trace(Sigma_p)
tr_Sigma_inv = np.trace(Sigma_p_inv)
d = 2

# q(x; theta) = N(mu_q, var_q * I)
mu0 = np.array([-4.0, 3.2])
log_var0 = np.log(4.5)   # optimize log(var) so variance stays positive

STEPS = 75
LR_MU_FORWARD = 0.8
LR_S_FORWARD = 0.3
LR_MU_REVERSE = 1.2
LR_S_REVERSE = 0.3
N_POINTS = 220
RNG = np.random.default_rng(10)

base_q = RNG.standard_normal((N_POINTS, 2))
base_p = RNG.standard_normal((N_POINTS, 2))
L_p = np.linalg.cholesky(Sigma_p)
target_points = mu_p + base_p @ L_p.T

def forward_kl(mu_q: np.ndarray, log_var: float) -> float:
    """D_KL(p || q) where p is full Gaussian and q is isotropic Gaussian."""
    var = np.exp(log_var)
    delta = mu_q - mu_p
    return 0.5 * ((tr_Sigma + delta @ delta) / var - d + d * log_var - logdet_Sigma)

def reverse_kl(mu_q: np.ndarray, log_var: float) -> float:
    """D_KL(q || p) where p is full Gaussian and q is isotropic Gaussian."""
    var = np.exp(log_var)
    delta = mu_q - mu_p
    return 0.5 * (var * tr_Sigma_inv + delta @ Sigma_p_inv @ delta - d + logdet_Sigma - d * log_var)

def grad_forward(mu_q: np.ndarray, log_var: float):
    var = np.exp(log_var)
    delta = mu_q - mu_p
    g_mu = delta / var
    A = tr_Sigma + delta @ delta
    g_s = 0.5 * (d - A / var)
    return g_mu, g_s

def grad_reverse(mu_q: np.ndarray, log_var: float):
    var = np.exp(log_var)
    delta = mu_q - mu_p
    g_mu = Sigma_p_inv @ delta
    g_s = 0.5 * (var * tr_Sigma_inv - d)
    return g_mu, g_s

def run(mode: str):
    mus = [mu0.copy()]
    log_vars = [float(log_var0)]
    kls = [forward_kl(mu0, log_var0) if mode == "forward" else reverse_kl(mu0, log_var0)]

    mu_q = mu0.copy()
    log_var = float(log_var0)

    for _ in range(STEPS - 1):
        if mode == "forward":
            g_mu, g_s = grad_forward(mu_q, log_var)
            mu_q = mu_q - LR_MU_FORWARD * g_mu
            log_var = log_var - LR_S_FORWARD * g_s
            kl = forward_kl(mu_q, log_var)
        else:
            g_mu, g_s = grad_reverse(mu_q, log_var)
            mu_q = mu_q - LR_MU_REVERSE * g_mu
            log_var = log_var - LR_S_REVERSE * g_s
            kl = reverse_kl(mu_q, log_var)

        mus.append(mu_q.copy())
        log_vars.append(float(log_var))
        kls.append(float(kl))

    return np.array(mus), np.array(log_vars), np.array(kls)

def ellipse_from_cov(mu, cov, n_std=2.0):
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals = vals[order]
    vecs = vecs[:, order]
    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    width, height = 2 * n_std * np.sqrt(vals)
    return width, height, angle

traj_forward = run("forward")
traj_reverse = run("reverse")

all_q_centers = np.vstack([traj_forward[0], traj_reverse[0]])
max_q_std = float(np.sqrt(max(np.exp(traj_forward[1]).max(), np.exp(traj_reverse[1]).max())))
target_stds = np.sqrt(np.linalg.eigvalsh(Sigma_p))
pad = 3.2 * max(max_q_std, target_stds.max())
mins = np.minimum(all_q_centers.min(axis=0), mu_p) - pad
maxs = np.maximum(all_q_centers.max(axis=0), mu_p) + pad
xlim = (mins[0], maxs[0])
ylim = (mins[1], maxs[1])

def _draw_panel(ax, frame: int, mode: str, mus, log_vars, kls):
    title_left = "Mode covering" if mode == "forward" else "Mode seeking"
    kl_label = r"$D_{KL}(p\|q)$" if mode == "forward" else r"$D_{KL}(q\|p)$"

    ax.clear()
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.18)
    ax.set_xlabel("x")
    ax.set_ylabel("y")

    mu_q = mus[frame]
    var_q = float(np.exp(log_vars[frame]))
    q_points = mu_q + np.sqrt(var_q) * base_q

    # point clouds
    ax.scatter(
        target_points[:, 0], target_points[:, 1],
        s=18, alpha=0.35, c="C0", label="target p(x)"
    )
    ax.scatter(
        q_points[:, 0], q_points[:, 1],
        s=18, alpha=0.45, c="C1", label="fitted qθ(x)"
    )

    # means
    ax.scatter(*mu_p, s=90, marker="x", c="C0")
    ax.scatter(*mu_q, s=90, marker="x", c="C1")

    # 2-sigma ellipses
    wp, hp, ap = ellipse_from_cov(mu_p, Sigma_p, n_std=2.0)
    ax.add_patch(Ellipse(mu_p, wp, hp, angle=ap, fill=False, lw=2.0, ec="C0"))

    wq, hq, aq = ellipse_from_cov(mu_q, var_q * np.eye(2), n_std=2.0)
    ax.add_patch(Ellipse(mu_q, wq, hq, angle=aq, fill=False, lw=2.0, ec="C1"))

    ax.legend(loc="upper left", frameon=True)
    ax.set_title(
        f"{title_left}: minimize {kl_label}\n"
        f"step {frame+1}/{len(mus)}   KL={kls[frame]:.3f}   isotropic var={var_q:.3f}"
    )


def make_animation(mode: str, save_path: str):
    """
    mode:
        - 'forward' : only D_KL(p || q)
        - 'reverse' : only D_KL(q || p)
        - 'both'    : side-by-side forward and reverse in one row
    """
    if mode not in {"forward", "reverse", "both"}:
        raise ValueError("mode must be one of: 'forward', 'reverse', 'both'")

    if mode == "both":
        fig, axes = plt.subplots(1, 2, figsize=(12.8, 6.0), sharex=True, sharey=True)
        fig.subplots_adjust(wspace=0.18, top=0.86)

        mus_f, log_vars_f, kls_f = traj_forward
        mus_r, log_vars_r, kls_r = traj_reverse
        n_frames = max(len(mus_f), len(mus_r))

        def update(frame):
            # clamp in case lengths differ
            f_idx = min(frame, len(mus_f) - 1)
            r_idx = min(frame, len(mus_r) - 1)

            _draw_panel(axes[0], f_idx, "forward", mus_f, log_vars_f, kls_f)
            _draw_panel(axes[1], r_idx, "reverse", mus_r, log_vars_r, kls_r)

            fig.suptitle("KL asymmetry: mode covering vs mode seeking", fontsize=14)

        ani = FuncAnimation(fig, update, frames=n_frames, interval=300, repeat=False)
        ani.save(save_path, writer=PillowWriter(fps=6))
        plt.close(fig)
        return

    # single-panel behavior (same as before)
    mus, log_vars, kls = traj_forward if mode == "forward" else traj_reverse

    fig, ax = plt.subplots(figsize=(6.4, 6.0))
    fig.subplots_adjust(top=0.88)

    def update(frame):
        _draw_panel(ax, frame, mode, mus, log_vars, kls)

    ani = FuncAnimation(fig, update, frames=len(mus), interval=180, repeat=False)
    ani.save(save_path, writer=PillowWriter(fps=6))
    plt.close(fig)

In [ ]:
make_animation("forward", "kl_forward_mode_covering.gif")
make_animation("reverse", "kl_reverse_mode_seeking.gif")
make_animation("both", "kl_both.gif")
